# XGBoost Random Forest
*Experiment 1. Random Forest*

In [ ]:
import os
from enum import Enum
from typing import Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import clear_output
from tqdm.auto import tqdm

from oldutils.datasets import (
    ARTIFACTS_FOLDER,
    ROOT_FOLDER,
    STATIC_FEATURES,
    HydroFiles,
    HydroStaticFeaturesFiles,
    MeteoSeriesFeaturesFiles,
)
from oldutils.types import TimeRange

%matplotlib inline

In [7]:
import polars as pl
import dask.dataframe as dd
from pathlib import Path

## Datasets

In [ ]:
PATH_MERGED_DATASETS = ARTIFACTS_FOLDER / "merged_datasets"
FILENAME_TRAIN_IDS = "train_file_ids.csv"

In [22]:
COL_GAUGE_ID = "gauge_id"

In [17]:
HORIZON_HISTORY = TimeRange.YEAR
HORIZON_FORECAST = TimeRange.WEEK

### Load datasets

In [9]:
def load_train_subset() -> dd:
    train_df = pl.read_csv(PATH_MERGED_DATASETS / FILENAME_TRAIN_IDS)
    file_ids = train_df["file_id"].to_list()
    paths = [PATH_MERGED_DATASETS / (str(file_id) + ".parquet") for file_id in file_ids]
    return dd.read_parquet(paths)

In [12]:
ddf = load_train_subset()
ddf.head()

,date,prcp,t_max,t_mean,t_min,q_mm_day,lvl_sm,gauge_id
0,2008-01-01,0.184543,-36.468083,-38.569561,-40.554178,0.047203,131.0,1151
1,2008-01-02,0.420088,-31.148393,-34.593115,-37.576221,0.045645,132.0,1151
2,2008-01-03,2.527887,-29.196286,-29.795872,-31.042251,0.044086,132.0,1151
3,2008-01-04,1.288245,-29.178316,-31.346584,-33.175628,0.042750,132.0,1151
4,2008-01-05,1.096579,-30.258844,-31.564057,-33.546148,0.041192,132.0,1151


### Add lags

In [44]:
ddf_copy = ddf.copy()
ddf_copy.sort_values(by=["gauge_id", "date"])

,date,prcp,t_max,t_mean,t_min,q_mm_day,lvl_sm,gauge_id
npartitions=1072,,,,,,,,
,date32[day][pyarrow],float64,float64,float64,float64,float64,float64,int32
,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...


In [ ]:
ddf_copy[["t_max_2", "t_min_2"]] = ddf[["t_max", "t_min"]].groupby(ddf[COL_GAUGE_ID]).shift(2, meta={"t_max": ""})
ddf_copy[["t_max_2", "t_min_2"]]

/tmp/ipykernel_300426/3833607218.py:3: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .shift(func)
  After:  .shift(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .shift(func, meta=('x', 'f8'))            for series result

  ddf_copy[["t_max_2", "t_min_2"]] = ddf[["t_max", "t_min"]].groupby(ddf[COL_GAUGE_ID]).shift(2)


,date,prcp,t_max,t_mean,t_min,q_mm_day,lvl_sm,gauge_id,t_max_2,t_min_2
npartitions=1072,,,,,,,,,,
,date32[day][pyarrow],float64,float64,float64,float64,float64,float64,int32,float64,float64
,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...


In [60]:
# Define a function doing exactly the grouped shift
def group_shift(df):
    return df.groupby(COL_GAUGE_ID)[["t_max","t_min"]].shift(3)

# Create an empty DataFrame holding just the schema
# Here a dict of {name: dtype} is fine
meta = {"t_max": "f8", "t_min": "f8"}

ddf_copy = ddf.copy()
shifted = ddf.map_partitions(group_shift, meta=meta)
ddf_copy[["t_max_2","t_min_2"]] = shifted

In [61]:
ddf_copy.compute()

KeyError: 'gauge'

In [42]:
# Compute the sum of missing values for t_max and t_min grouped by COL_GAUGE_ID
result1 = ddf[["t_max", "t_min"]].isna().groupby(ddf[COL_GAUGE_ID]).sum().compute()

# Compute the sum of missing values for t_max and t_min after shifting by 2, grouped by COL_GAUGE_ID
result2 = ddf[["t_max", "t_min"]].groupby(ddf[COL_GAUGE_ID]).shift(2).compute().isna().sum().compute()

/tmp/ipykernel_300426/363862853.py:5: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .shift(func)
  After:  .shift(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .shift(func, meta=('x', 'f8'))            for series result

  result2 = ddf[["t_max", "t_min"]].groupby(ddf[COL_GAUGE_ID]).shift(2).compute().isna().sum().compute()


ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
(result1.sum(), result2.sum())


(t_max    0
 t_min    0
 dtype: int64,
 t_max    2
 t_min    2
 dtype: int64)

In [ ]:
def add_lags(ddf, column, lags=HORIZON_HISTORY):
    

### Add static features

In [14]:
hsff = HydroStaticFeaturesFiles()
for i in hsff:
    print(i[1].collect())
    break

shape: (1, 150)
┌──────────┬────────────┬────────────┬────────────┬───┬────────────┬────────────┬───────────┬──────┐
│ gauge_id ┆ for_pc_sse ┆ crp_pc_sse ┆ inu_pc_ult ┆ … ┆ ero_kh_sav ┆ hft_ix_s93 ┆ hft_ix_s0 ┆ acc  │
│ ---      ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---        ┆ 9         ┆ ---  │
│ i64      ┆ f64        ┆ f64        ┆ f64        ┆   ┆ f64        ┆ f64        ┆ ---       ┆ f64  │
│          ┆            ┆            ┆            ┆   ┆            ┆            ┆ f64       ┆      │
╞══════════╪════════════╪════════════╪════════════╪═══╪════════════╪════════════╪═══════════╪══════╡
│ 1001     ┆ 56.666633  ┆ 0.0        ┆ 5.804287   ┆ … ┆ null       ┆ null       ┆ null      ┆ null │
└──────────┴────────────┴────────────┴────────────┴───┴────────────┴────────────┴───────────┴──────┘
